In [73]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import settings
from src.aggregations import territorial, person, labor,cadunico
import pandas as pd 
import numpy as np
from pathlib import Path
import importlib

importlib.reload(settings)
importlib.reload(territorial)
importlib.reload(person)
importlib.reload(labor)
importlib.reload(cadunico)



pd.set_option("display.max_columns", None)

In [2]:
df_cubo = pd.read_excel(settings.CUBO_PATH)

In [37]:
df_cubo_x = pd.read_excel(settings.DATA_PATH /'input_data' /'dados_cubo_final_v0__2026-05-28_16-48.xlsx')

In [ ]:
# Section 1
# bignumber1.csv

# Valor executado por estado x municipio abs / percentual 
df_cubo_est = df_cubo[df_cubo['tipo_ente'] == 'ESTADO']
df_cubo_mun = df_cubo[df_cubo['tipo_ente'] == 'MUNICIPIO']

# Totais executados com ceil
valor_estados = np.ceil(df_cubo_est["valor_transacao"].sum())
valor_municipios = np.ceil(df_cubo_mun["valor_transacao"].sum())

# Valores de referência
total_estados = 1_510_000_000
total_municipios = 1_490_000_000

df_execucao = pd.DataFrame({
    "Estados_DF": [valor_estados],
    "Municipios_DF": [valor_municipios],
    "perc_executado_estados": [valor_estados / total_estados],
    "perc_executado_municipios": [valor_municipios / total_municipios],
})

df_execucao

,Estados_DF,Municipios_DF,perc_executado_estados,perc_executado_municipios
0,1.450514e+09,1.395481e+09,0.960606,0.936565


In [42]:
df_execucao.to_csv('s1_bn1.csv')

In [36]:
# executed_value_by.csv
df_states       = territorial.executed_value_n_contemplados_qty_by(df_cubo=df_cubo, by_filter='ESTADO')
df_municipality = territorial.executed_value_n_contemplados_qty_by(df_cubo=df_cubo, by_filter='MUNICIPIO')
df_uf           = territorial.executed_value_n_contemplados_qty_by(df_cubo=df_cubo, by_filter='UF')

df_states.to_csv(settings.DATA_PATH_SECTION1 / 'executed_value_by_state.csv')
df_municipality.to_csv(settings.DATA_PATH_SECTION1 / 'executed_value_by_municipality.csv')
df_uf.to_csv(settings.DATA_PATH_SECTION1 / 'executed_value_by_uf.csv')

In [22]:
# executed_value_by_region
df_states_region       = territorial.aggregate_execution_by_region(df_cubo=df_cubo, by_filter='ESTADO')
df_municipality_region = territorial.aggregate_execution_by_region(df_cubo=df_cubo, by_filter='MUNICIPIO')
df_uf_region           = territorial.aggregate_execution_by_region(df_cubo=df_cubo, by_filter='UF')

df_states_region.to_csv(settings.DATA_PATH_SECTION1 / 'executed_value_by_region_state.csv')
df_municipality_region.to_csv(settings.DATA_PATH_SECTION1 / 'executed_value_by_region_municipality.csv')
df_uf_region.to_csv(settings.DATA_PATH_SECTION1 / 'executed_value_by_region_uf.csv')


In [25]:
# aggregate_values_by
df_municipality_agg     = territorial.aggregate_execution_summary_by_scope(df_cubo=df_cubo, scope='MUNICIPIO')
df_state_agg            = territorial.aggregate_execution_summary_by_scope(df_cubo=df_cubo, scope='ESTADO')
df_capital_agg          = territorial.aggregate_capital_interior_summary(df_cubo=df_cubo)

df_municipality_agg.to_csv(settings.DATA_PATH_SECTION1 / 'aggregate_values_by_municipality.csv')
df_capital_agg.to_csv(settings.DATA_PATH_SECTION1 / 'aggregate_values_by_capital.csv')
df_state_agg.to_csv(settings.DATA_PATH_SECTION1 / 'aggregate_values_by_state.csv')

In [26]:
df_capital_agg

,valor_total_capital,quantidade_total_capital,percentual_valor_capital,percentual_quantidade_capital,quantidade_feminino_capital,percentual_feminino_capital,quantidade_masculino_capital,percentual_masculino_capital,valor_total_interior,quantidade_total_interior,percentual_valor_interior,percentual_quantidade_interior,quantidade_feminino_interior,percentual_feminino_interior,quantidade_masculino_interior,percentual_masculino_interior
0,282078707.67,6167,0.202137,0.042579,1867,0.30274,2087,0.338414,1113402560.22,138669,0.797863,0.957421,53975,0.389236,61672,0.444743


In [ ]:
df_cubo[df_cubo['flag_'] == 'Rural']['valor_transacao'].sum()

np.float64(176056645.95)

In [11]:
# values_by_population_size
df_population_size = territorial.aggregate_execution_by_porte_with_estado(df_cubo=df_cubo)
df_population_size.to_csv(settings.DATA_PATH_SECTION1 / 'values_by_population_size.csv')

In [19]:
denominador_urbano_rural = (
    df_population_size["valor_urbano_por_porte"]
    + df_population_size["valor_rural_por_porte"]
)

df_population_size["percentual_valor_urbano_por_porte"] = np.where(
    denominador_urbano_rural.ne(0),
    df_population_size["valor_urbano_por_porte"] / denominador_urbano_rural,
    np.nan
)

df_population_size["percentual_valor_rural_por_porte"] = np.where(
    denominador_urbano_rural.ne(0),
    df_population_size["valor_rural_por_porte"] / denominador_urbano_rural,
    np.nan
)

In [20]:
df_population_size.to_csv(settings.DATA_PATH_SECTION1 / 'values_by_population_size.csv')

In [17]:
147500752.72/(147500752.72+12718073.4)

0.920620605530623

In [4]:
df_population_size_mean = territorial.resumo_valor_por_porte_municipio(df_cubo=df_cubo)

In [6]:
df_population_size_mean.to_csv(settings.DATA_PATH_SECTION1 / 'population_size_mean.csv')

In [74]:
# values_by_special_territory
df_special_territory_municipality = territorial.aggregate_special_territories_by(
    df_cubo=df_cubo, 
    categories=settings.CATEGORIES_SPECIAL_TERRITORIES, 
    by_filter="MUNICIPIO"
)

df_special_territory_state = territorial.aggregate_special_territories_by(
    df_cubo=df_cubo, 
    categories=settings.CATEGORIES_SPECIAL_TERRITORIES, 
    by_filter="ESTADO"
)

df_special_territory_uf = territorial.aggregate_special_territories_by(
    df_cubo=df_cubo, 
    categories=settings.CATEGORIES_SPECIAL_TERRITORIES, 
    by_filter="UF"
)

df_special_territory_municipality.to_csv(settings.DATA_PATH_SECTION1 / 'values_by_special_territory_municipality.csv')
df_special_territory_state.to_csv(settings.DATA_PATH_SECTION1 / 'values_by_special_territory_state.csv')
df_special_territory_uf.to_csv(settings.DATA_PATH_SECTION1 / 'values_by_special_territory_uf.csv')


In [72]:
df_special_territory_uf

,cod_tipo_nome,valor_transacao,perc_valor_transacao,quantidade_contemplados,perc_quantidade_contemplados,qtd_tipo_documento_CNPJ,qtd_tipo_documento_CPF,valor_tipo_documento_CNPJ,valor_tipo_documento_CPF,min_valor_tipo_documento_CNPJ,min_valor_tipo_documento_CPF,mediana_valor_tipo_documento_CNPJ,mediana_valor_tipo_documento_CPF,max_valor_tipo_documento_CNPJ,max_valor_tipo_documento_CPF,media_valor_tipo_documento_CNPJ,media_valor_tipo_documento_CPF
0,Não especial,2653700009,0.9324,160271,0.9604,30918,129353,1479002422,1174697588,375,375,12941,3572,19987818,735000,61086,9395
1,Favela e Comunidade Urbana,138146185,0.0485,4555,0.0273,864,3691,76558971,61587214,431,400,36000,10000,22109765,400000,109214,16874
2,Setor com baixo patamar domiciliar,21852365,0.0077,673,0.0040,260,413,15226973,6625393,406,378,20000,5000,1031144,650000,59714,16082
3,Agrupamento quilombola,7369210,0.0026,481,0.0029,72,409,3509108,3860103,1034,500,29307,3200,329600,150000,52375,9532
4,Agrupamento indígena,5078944,0.0018,254,0.0015,22,232,1391182,3687762,4000,500,25883,5500,250000,294095,63236,16175
5,Quartel e base militar,480603,0.0002,24,0.0001,1,23,300000,180603,300000,1089,300000,3842,300000,50000,300000,7853
6,Não informado,18385602,0.0065,534,0.0032,135,399,15166170,3219433,380,500,20000,3000,2017102,330000,156353,8384
7,Agrovila do PA,540519,0.0002,63,0.0004,4,59,66000,474519,6000,500,20000,3334,20000,127223,16500,8043
8,Unidade prisional,86969,0.0000,10,0.0001,1,9,6870,80099,6870,800,6870,6000,6870,26880,6870,8900
9,Convento / hospital / ILPI / IACA,152685,0.0001,13,0.0001,3,10,84000,68685,4000,2000,20000,5705,60000,15000,28000,6869


In [66]:
df_cubo.head(2)

,ente,tipo_ente,tipo_documento,faixa_vlr_pago,uf,nome_ente,regiao,flag_capital,porte_populacional,Sexo,Estrangeiro,NomeNaturezaOcupacao,NomeOcupacaoPrincipal,faixa_etaria,flag_cpf_mei,cnaePrincipal,naturezaJuridica,porte,cnpj_optante_mei,raca_cor_desc_description,escolaridade_description,ind_deficiencia,tipo_deficiencia_description,tipo_vinculo_description,faixa_salarial_rais,CBO_2002_RAIS,cbo_codigo,cbo_descricao,SITUACAO,cod_situacao_nome,cod_tipo_nome,pessoaCad_cadunico,familiaPBF_cadunico,fxRendaFamiliarTotal_desc_cadunico,caracDomicilio_desc_cadunico,fxRendaPerCapita_desc_cadunico,pertence_bpc,categoria_municipio_ibge,faixa_vlr_pago_ju_bbagil,situacao_renda_cadunico,cod_cnae_principal_receita_cnpj,descr_cnae_principal_receita_cnpj,naturezajuridica_agrupada_receita_cnpj,flag_cnae_cultural,tipo_vinculo_agregado_rais,escolaridade_agregado_rais,flag_cbo_cultural_rais,flag_join_rais,quantidade,valor_transacao,min_valor_transacao,max_valor_transacao,sum_populacao
0,AC_Acre_12,ESTADO,CNPJ,1 milhão a 10 milhões,AC,Acre,Norte,False,-99,NaN,NaN,NaN,NaN,NaN,False,"{'codigo': '7820500', 'descricao': 'Locação de...","{'codigo': '2062', 'descricao': 'Sociedade Emp...",01 - MicroEmpresa-ME,0.0,NaN,NaN,NaN,NaN,NaN,NÃO SE APLICA,NaN,NaN,NaN,Urbana,Área urbana de alta densidade de edificações d...,Não especial,NaN,NaN,NaN,NaN,NaN,NaN,Interior,Acima de 200 mil,NaN,7820500.0,Locação de mão-de-obra temporária,Entidades empresariais | EPP | Microempresa,CNAE NAO CULTURAL,NaN,NaN,NaN,False,1,2034002.65,2034002.65,2034002.65,880631
1,AC_Acre_12,ESTADO,CNPJ,10 a 50 mil,AC,Acre,Norte,False,-99,NaN,NaN,NaN,NaN,NaN,False,"{'codigo': '1629301', 'descricao': 'Fabricação...","{'codigo': '2135', 'descricao': 'Empresário (I...",01 - MicroEmpresa-ME,1.0,NaN,NaN,NaN,NaN,NaN,NÃO SE APLICA,NaN,NaN,NaN,Urbana,Área urbana de alta densidade de edificações d...,Não especial,NaN,NaN,NaN,NaN,NaN,NaN,Interior,De 10 a 50 mil,NaN,1629301.0,"Fabricação de artefatos diversos de madeira, e...",Entidades empresariais | EPP | Microempresa,CNAE CULTURAL,NaN,NaN,NaN,False,1,50000.00,50000.00,50000.00,880631


In [64]:
# special_territory_w_ibge_by_brazil
df_vis_territorio_brasil = territorial.generate_special_territories_brazil_view(df_cubo=df_cubo)
df_vis_territorio_brasil.to_csv(settings.DATA_PATH_SECTION1 / 'special_territory_w_ibge_by_brazil.csv')

In [52]:
# aggregate_by_local_residencia
df_interior_rm_uf = territorial.aggregate_by_local_residencia(df_cubo=df_cubo_x, visao='uf')
df_interior_rm_uf.to_csv(settings.DATA_PATH_SECTION1 / 'aggregate_by_local_residencia_uf.csv')

In [61]:
df_cubo.head(2)

,ente,tipo_ente,tipo_documento,faixa_vlr_pago,uf,nome_ente,regiao,flag_capital,porte_populacional,Sexo,Estrangeiro,NomeNaturezaOcupacao,NomeOcupacaoPrincipal,faixa_etaria,flag_cpf_mei,cnaePrincipal,naturezaJuridica,porte,cnpj_optante_mei,raca_cor_desc_description,escolaridade_description,ind_deficiencia,tipo_deficiencia_description,tipo_vinculo_description,faixa_salarial_rais,CBO_2002_RAIS,cbo_codigo,cbo_descricao,SITUACAO,cod_situacao_nome,cod_tipo_nome,pessoaCad_cadunico,familiaPBF_cadunico,fxRendaFamiliarTotal_desc_cadunico,caracDomicilio_desc_cadunico,fxRendaPerCapita_desc_cadunico,pertence_bpc,categoria_municipio_ibge,faixa_vlr_pago_ju_bbagil,situacao_renda_cadunico,cod_cnae_principal_receita_cnpj,descr_cnae_principal_receita_cnpj,naturezajuridica_agrupada_receita_cnpj,flag_cnae_cultural,tipo_vinculo_agregado_rais,escolaridade_agregado_rais,flag_cbo_cultural_rais,flag_join_rais,quantidade,valor_transacao,min_valor_transacao,max_valor_transacao,sum_populacao
0,AC_Acre_12,ESTADO,CNPJ,1 milhão a 10 milhões,AC,Acre,Norte,False,-99,NaN,NaN,NaN,NaN,NaN,False,"{'codigo': '7820500', 'descricao': 'Locação de...","{'codigo': '2062', 'descricao': 'Sociedade Emp...",01 - MicroEmpresa-ME,0.0,NaN,NaN,NaN,NaN,NaN,NÃO SE APLICA,NaN,NaN,NaN,Urbana,Área urbana de alta densidade de edificações d...,Não especial,NaN,NaN,NaN,NaN,NaN,NaN,Interior,Acima de 200 mil,NaN,7820500.0,Locação de mão-de-obra temporária,Entidades empresariais | EPP | Microempresa,CNAE NAO CULTURAL,NaN,NaN,NaN,False,1,2034002.65,2034002.65,2034002.65,880631
1,AC_Acre_12,ESTADO,CNPJ,10 a 50 mil,AC,Acre,Norte,False,-99,NaN,NaN,NaN,NaN,NaN,False,"{'codigo': '1629301', 'descricao': 'Fabricação...","{'codigo': '2135', 'descricao': 'Empresário (I...",01 - MicroEmpresa-ME,1.0,NaN,NaN,NaN,NaN,NaN,NÃO SE APLICA,NaN,NaN,NaN,Urbana,Área urbana de alta densidade de edificações d...,Não especial,NaN,NaN,NaN,NaN,NaN,NaN,Interior,De 10 a 50 mil,NaN,1629301.0,"Fabricação de artefatos diversos de madeira, e...",Entidades empresariais | EPP | Microempresa,CNAE CULTURAL,NaN,NaN,NaN,False,1,50000.00,50000.00,50000.00,880631


# Section 2

In [27]:
df_values_by_person_type_uf = territorial.aggregate_execution_by_person_type(df_cubo=df_cubo, by_filter='UF')
df_values_by_person_type_state = territorial.aggregate_execution_by_person_type(df_cubo=df_cubo, by_filter='ESTADO')
df_values_by_person_type_municipality = territorial.aggregate_execution_by_person_type(df_cubo=df_cubo, by_filter='MUNICIPIO')

df_values_by_person_type_uf.to_csv(settings.DATA_PATH_SECTION2 / 'aggregate_execution_by_person_type_uf.csv')
df_values_by_person_type_state.to_csv(settings.DATA_PATH_SECTION2 / 'aggregate_execution_by_person_type_state.csv')
df_values_by_person_type_municipality.to_csv(settings.DATA_PATH_SECTION2 / 'aggregate_execution_by_person_type_municipality.csv')

# Section 3

In [16]:
importlib.reload(person)

<module 'src.aggregations.person' from 'c:\\Users\\gabiru\\Documents\\GitHub\\pnab-data-vis\\src\\aggregations\\person.py'>

In [7]:
# aggregate_contemplados_pf_pj_proportion.csv
df_person = person.aggregate_contemplados_pf_pj_proportion(df_cubo=df_cubo)
df_person.to_csv(settings.DATA_PATH_SECTION3 / 'aggregate_contemplados_pf_pj_proportion.csv')

In [11]:
df_cubo['tipo_documento'].value_counts(dropna=False)

tipo_documento
CPF     130235
CNPJ     25363
Name: count, dtype: int64

In [8]:
# aggregate_contemplados_by_sexo_proportion.csv
df_sexo = person.aggregate_contemplados_by_sexo_proportion(df_cubo=df_cubo)
df_sexo.to_csv(settings.DATA_PATH_SECTION3 / 'aggregate_contemplados_by_sexo_proportion.csv')

In [50]:
df_sexo.head(2)

,quantidade_contemplados,perc_quantidade_contemplados,valor_contemplados,perc_valor_contemplados,quantidade_contemplados_feminino,perc_quantidade_contemplados_feminino,valor_contemplados_feminino,perc_valor_contemplados_feminino,quantidade_contemplados_masculino,perc_quantidade_contemplados_masculino,valor_contemplados_masculino,perc_valor_contemplados_masculino
0,134593,1,1254423238,1,62943,0.467654,578195818,0.460926,71650,0.532346,676227421,0.539074


In [13]:
# aggregate_valor_quantity_by_age_group_sexo_wide
df_age_group = person.aggregate_valor_quantity_by_age_group_sexo_wide(df_cubo=df_cubo)
df_age_group.to_csv(settings.DATA_PATH_SECTION3 / 'aggregate_valor_quantity_by_age_group_sexo_wide.csv')

In [ ]:
# aggregate_value_quantity_by_age_group_region_wide
df_age_region = person.aggregate_value_quantity_by_age_group_region_wide(df_cubo=df_cubo)
df_age_region.to_csv(settings.DATA_PATH_SECTION3 / 'aggregate_value_quantity_by_age_group_region_wide.csv')

# Section 4

In [43]:
importlib.reload(labor)

<module 'src.aggregations.labor' from 'c:\\Users\\gabiru\\Documents\\GitHub\\pnab-data-vis\\src\\aggregations\\labor.py'>

In [41]:
# aggregate_vinculo_formal_labor.csv
df_not_in_mercado = labor.aggregate_vinculo_formal_labor(df_cubo=df_cubo)
df_not_in_mercado.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor.csv')

In [42]:
# aggregate_vinculo_formal_labor_by_uf.csv
df_not_in_mercado_by_uf = labor.aggregate_vinculo_formal_labor_by_uf(df_cubo=df_cubo)
df_not_in_mercado_by_uf.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor_by_uf.csv')

In [43]:
# aggregate_vinculo_formal_labor_by_region.csv
df_not_in_mercado_by_region = labor.aggregate_vinculo_formal_labor_by_region(df_cubo=df_cubo)
df_not_in_mercado_by_region.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor_by_region.csv')


In [46]:
# aggregate_vinculo_formal_labor_by_sexo.csv
df_not_in_mercado_by_sexo = labor.aggregate_vinculo_formal_labor_by_sexo(df_cubo=df_cubo)
df_not_in_mercado_by_sexo.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor_by_sexo.csv')

In [50]:
# aggregate_vinculo_formal_labor_by_age_group.csv
df_not_in_mercado_by_age_group = labor.aggregate_vinculo_formal_labor_by_age_group(df_cubo=df_cubo)
df_not_in_mercado_by_age_group.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor_by_age_group.csv')

In [56]:
# aggregate_vinculo_formal_labor_by_raca_cor.csv
df_not_in_mercado_by_raca_cor = labor.aggregate_vinculo_formal_labor_by_raca_cor(df_cubo=df_cubo)
df_not_in_mercado_by_raca_cor.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor_by_raca_cor.csv')


In [ ]:
# aggregate_raca_cor_vinculo_formal_labor_by_sexo
df_not_in_mercado_by_raca_cor_sexo = labor.aggregate_raca_cor_vinculo_formal_labor_by_sexo(df_cubo=df_cubo)
df_not_in_mercado_by_raca_cor_sexo.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_raca_cor_vinculo_formal_labor_by_sexo.csv')

In [67]:
# aggregate_vinculo_formal_labor_by_escolaridade.csv
df_not_in_mercado_escolaridade = labor.aggregate_vinculo_formal_labor_by_escolaridade(df_cubo=df_cubo)
df_not_in_mercado_escolaridade.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_formal_labor_by_escolaridade.csv')

In [69]:
# aggregate_vinculo_trabalho_formal_by_escolaridade_clean.csv
df_not_in_mercado_escolaridade_clean = labor.aggregate_vinculo_trabalho_formal_by_escolaridade_sem_sem_informacao(df_cubo=df_cubo)
df_not_in_mercado_escolaridade_clean.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_vinculo_trabalho_formal_by_escolaridade_clean.csv')

In [ ]:
# CBOS

np.float64(12415168.75)

In [46]:
# aggregate_cbo_rais.csv
df_cbo_rais = labor.aggregate_cbo_rais(df_cubo=df_cubo)
df_cbo_rais.to_csv(settings.DATA_PATH_SECTION4 / 'aggregate_cbo_rais.csv')

# Section 5

In [177]:
importlib.reload(cadunico)

<module 'src.aggregations.cadunico' from 'c:\\Users\\gabiru\\Documents\\GitHub\\pnab-data-vis\\src\\aggregations\\cadunico.py'>

In [47]:
df_cad_unico = pd.read_parquet(settings.DATA_PATH / 'input_data' / 'non-public' /'dim_cadunico__2026-05-19_18-17.parquet')

In [100]:
# aggregate_cadunico_summary.csv
df_cubo_cadunico = cadunico.aggregate_cadunico_summary(df_cubo=df_cubo)
df_cubo_cadunico.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_summary.csv')

In [101]:
# aggregate_cadunico_profile_summary.csv
df_cubo_cadunico_sexo_idade = cadunico.aggregate_cadunico_profile_summary(df_cubo=df_cubo)
sexo = df_cubo_cadunico_sexo_idade[df_cubo_cadunico_sexo_idade['dimensao'] == 'Sexo']
faixa_etaria = df_cubo_cadunico_sexo_idade[df_cubo_cadunico_sexo_idade['dimensao'] == 'Faixa etária']
sexo.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_profile_summary_by_sexo.csv')
faixa_etaria.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_profile_summary_by_faixa_etaria.csv')


In [ ]:
# aggregate_cadunico_faixa_etaria_by_sexo.csv
df_cubo_cadunico_sexo_idade_juntos = cadunico.aggregate_cadunico_faixa_etaria_by_sexo(df_cubo=df_cubo)
df_cubo_cadunico_sexo_idade_juntos.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_faixa_etaria_by_sexo.csv')

In [ ]:
# aggregate_cadunico_by_situacao_renda.csv
df_cad_unico_situacao_renda = cadunico.aggregate_cadunico_by_situacao_renda(df_cubo=df_cubo)
df_cad_unico_situacao_renda.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_by_situacao_renda.csv')


In [ ]:
# aggregate_cadunico_by_fx_renda_per_capita.csv
df_cad_unico_situacao_faixa_renda_percapita = cadunico.aggregate_cadunico_by_fx_renda_per_capita(df_cubo=df_cubo)
df_cad_unico_situacao_faixa_renda_percapita.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_by_fx_renda_per_capita.csv')

In [ ]:
# aggregate_cadunico_by_situacao_domicilio.csv
df_cad_unico_domicilio_situacao = cadunico.aggregate_cadunico_by_situacao_domicilio(df_cubo=df_cubo)
df_cad_unico_domicilio_situacao.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_by_situacao_domicilio.csv')

In [187]:
# aggregate_cadunico_by_population_size.csv
df_cad_unico_porte_populacional = cadunico.aggregate_cadunico_by_population_size(df_cubo=df_cubo)
df_cad_unico_porte_populacional.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_by_population_size.csv')


In [188]:
# aggregate_cadunico_by_uf.csv
df_cad_unico_by_uf = cadunico.aggregate_cadunico_by_uf(df_cubo=df_cubo)
df_cad_unico_by_uf.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_by_uf.csv')

In [172]:
# aggregate_cadunico_by_value_group.csv
df_cad_unic_faixa_valor = cadunico.aggregate_cadunico_by_value_group(df_cubo=df_cubo)
df_cad_unic_faixa_valor.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_cadunico_by_value_group.csv')


In [173]:
# aggregate_bolsa_familia_summary.csv
df_cad_unico_bpf = cadunico.aggregate_bolsa_familia_summary(df_cubo=df_cubo)
df_cad_unico_bpf.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_bolsa_familia_summary.csv')

In [180]:
# aggregate_bpc_summary.csv
df_cad_unico_bpc = cadunico.aggregate_bpc_summary(df_cubo=df_cubo)
df_cad_unico_bpc.to_csv(settings.DATA_PATH_SECTION5 / 'aggregate_bpc_summary.csv')

In [181]:
df_cpf_receita = pd.read_parquet(settings.DATA_PATH / 'input_data' / 'non-public' /'dim_cpf_receita__2026_05-19-13_09.parquet')

In [183]:
df_cpf_receita[df_cpf_receita['sexo_receita_cpf'] == 'Feminino']['cpf_receita_cpf'].nunique()

61026

In [184]:
df_cpf_receita[df_cpf_receita['sexo_receita_cpf'] == 'Masculino']['cpf_receita_cpf'].nunique()

68615

In [186]:
61026/(61026+68615)

0.47073071019199175